## 前処理(実行済み)


### データセットについて
URMPのwavデータセットをINPUT_DIRに保存。全13の楽器で、様々な編成により44曲の演奏が行われたもので、個別の演奏データも記録されている。本研究ではその個別データを扱う。

### 前処理について
各楽器の簡略名(cl, bn, vn等)のディレクトリに振り分けられた個別データについて以下の操作を行う
1. 最もデータ数の少ないdbに合わせ、3つのファイルを選択(google driveのソート順で選択。ランダム性はなし)。計3分程度の音声ファイルに対して前処理を行う。
2. 各音声ファイルを4秒ごとのチャンクに分割し、各チャンクに対して「元音源」「A-weightingに基づいたloudness」「crepeの最も精度が高いモデルによる基本周波数(f0)」「対数メルスペクトログラム」「楽器名（略記）」を保持した.ptファイルを生成し、OUTPUT_DIRに保存する

In [ ]:
# データの場所
# 構成例: raw_data/Violin/01.wav, raw_data/Flute/song.wav ...
INPUT_DIR = "drive/MyDrive/hioki_lab/data/URMP_solo_inst/AuSep"
OUTPUT_DIR = "drive/MyDrive/hioki_lab/data/URMP_solo_inst/processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# デバイス（CREPE用）
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# === 関数定義 ===

def compute_loudness(audio, sample_rate, n_fft=2048, hop_length=64):
    """A-weightingされたLoudnessを計算"""
    stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    power = np.abs(stft)**2
    freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=n_fft)
    a_weights = librosa.A_weighting(freqs)
    a_weights = 10**(a_weights/10)
    a_weights = a_weights[:, np.newaxis]

    weighted_power = np.sum(power * a_weights, axis=0)
    loudness = librosa.power_to_db(weighted_power, ref=np.max)
    return loudness

def extract_features(wav_path, instrument_name, chunk_length_samples):
    """1つのWAVファイルを読み込み、分割して保存する"""
    # 1. Load Audio
    audio, sr = torchaudio.load(wav_path)
    # ステレオならモノラル化
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    # リサンプリング
    if sr != SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
        audio = resampler(audio)

    audio = audio.squeeze().numpy() # numpyへ

    # 無音除去（簡易的）
    # あまりに静かな部分は学習の邪魔になるのでトリミングすると良いですが
    # ここではシンプルに進めます

    # 2. Chunking (4秒ごとに切り出し)
    total_samples = len(audio)
    num_chunks = total_samples // chunk_length_samples

    print(f"Processing {wav_path} -> {num_chunks} chunks...")

    for i in range(num_chunks):
        start = i * chunk_length_samples
        end = start + chunk_length_samples
        audio_chunk = audio[start:end]

        # --- Tensor化 (GPUへ送る準備) ---
        audio_tensor = torch.from_numpy(audio_chunk).unsqueeze(0).to(device)

        # 3. CREPEによるf0抽出
        # fullモデルを使うと精度が良い
        f0 = torchcrepe.predict(
            audio_tensor,
            SAMPLE_RATE,
            hop_length=HOP_LENGTH,
            fmin=50,
            fmax=2000,
            model='full',
            batch_size=1,
            device=device
        )
        f0 = f0.squeeze().cpu().float() # [Time] 。squeezeで余分な次元を削除

        # f0の信頼度が低いところ（無音など）は0にするマスク処理も重要ですが
        # VAEの場合は「無音であること」も学習させたいのでそのままにします

        # 4. Loudness抽出
        loudness = compute_loudness(audio_chunk, SAMPLE_RATE, N_FFT, HOP_LENGTH)
        # CREPEとフレーム数が微妙にずれることがあるので合わせる
        min_len = min(len(f0), len(loudness))
        f0 = f0[:min_len]
        loudness = torch.from_numpy(loudness[:min_len]).float()

        # 5. Mel-Spectrogram (Encoder入力用)
        # torchaudioを使用
        mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            f_min=20,     # 低域ノイズカット
            f_max=8000,   # ナイキストまで
            power=1.0     # エネルギーではなく振幅スペクトル
        ).to(device)

        mel = mel_transform(audio_tensor) # [1, n_mels, Time]
        mel = torch.log(torch.clamp(mel, min=1e-5)) # Log-Melにする (聴感補正、学習効率の向上)

        # 時間軸のサイズ合わせ (Melとf0/loudnessはずれる可能性あり)
        # Encoder入力(Mel)は少し長くても良いが、Decoder入力(f0)と合わせるためにトリミング
        mel = mel[:, :, :min_len].cpu()

        # 6. 保存
        save_name = f"{instrument_name}_{os.path.basename(wav_path).replace('.wav', '')}_{i:03d}.pt"
        save_path = os.path.join(OUTPUT_DIR, save_name)

        torch.save({
            'audio': torch.from_numpy(audio_chunk[:min_len * HOP_LENGTH]), # Audio
            'mel': mel,                 # [1, 80, T] -> VAE Input
            'f0': f0.unsqueeze(-1),     # [T, 1] -> DDSP Input
            'loudness': loudness.unsqueeze(-1), # [T, 1] -> DDSP Input
            'instrument_name': instrument_name
        }, save_path)

# === 実行部分 ===
# raw_data/Violin/xxx.wav のような構造を想定
def run_preprocessing():
    chunk_samples = int(CHUNK_SEC * SAMPLE_RATE)

    print(f"Looking for data in: {os.path.abspath(INPUT_DIR)}")

    # 楽器フォルダごとに処理
    inst_dirs = glob.glob(os.path.join(INPUT_DIR, "*"))
    print(f"Found directories: {inst_dirs}")
    for d in inst_dirs:
        if not os.path.isdir(d): continue
        inst_name = os.path.basename(d)

        wav_files = glob.glob(os.path.join(d, "*.wav"))
        if len(wav_files) > MAX_FILES_PER_INST:
            wav_files = wav_files[:MAX_FILES_PER_INST]

        print(f"Found {len(wav_files)} files in {inst_name}")

        for w in wav_files:
            extract_features(w, inst_name, chunk_samples)

        print(f"Done: {inst_name}")
if __name__ == "__main__":
    # 事前に raw_data フォルダを作り、その中に Violin フォルダ等を作ってWAVを入れてください
    print("Starting Preprocessing... (Takes time because of CREPE)")
    run_preprocessing()
    print("Done!")